# 08 - Answer Synthesis 및 Output Mode

이 노트북에서는 **Agentic Retrieval**의 **Output Mode**와 **Answer Synthesis** 기능을 학습합니다.

## 📋 학습 내용
1. **Output Mode 비교** - Extractive Data vs Answer Synthesis
2. **Answer Instructions** - 답변 생성 지침 커스터마이징
3. **응답 형식 제어** - 구조화된 출력 생성
4. **도메인별 커스터마이징** - 사용 사례별 최적화

## ⚠️ 사전 요구사항
- `01-setup_knowledge_base.ipynb` 실행 완료

## 1. 환경 설정

In [ ]:
import os
import json
import time
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import KnowledgeBaseRetrievalClient
from azure.search.documents.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalMediumReasoningEffort,
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalAnswerInstructions,
)

# 환경 변수 로드
load_dotenv()

# Azure AI Search 설정
search_endpoint = os.environ["SEARCH_ENDPOINT"]
search_api_key = os.environ["SEARCH_ADMIN_KEY"]
search_credential = AzureKeyCredential(search_api_key)

# 리소스 이름
KNOWLEDGE_BASE_NAME = "products-kb"

# 클라이언트 생성
kb_client = KnowledgeBaseRetrievalClient(
    endpoint=search_endpoint,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential
)

print(f"✅ 연결 완료: {search_endpoint}")
print(f"✅ Knowledge Base: {KNOWLEDGE_BASE_NAME}")

## 2. Output Mode 개요

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                          Output Mode 비교                                     ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  📊 EXTRACTIVE_DATA (추출 모드)                                               ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │ • 원본 문서 청크를 그대로 반환                                           │  ║
║  │ • LLM이 답변을 생성하지 않음                                             │  ║
║  │ • 빠른 응답, 낮은 비용                                                   │  ║
║  │ • 클라이언트에서 답변 생성 필요                                          │  ║
║  │                                                                        │  ║
║  │ 적합한 경우:                                                            │  ║
║  │   - RAG 파이프라인에서 검색만 필요                                       │  ║
║  │   - 클라이언트에서 자체 LLM 사용                                         │  ║
║  │   - 원본 데이터 정확도가 중요                                            │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
║  📝 ANSWER_SYNTHESIS (합성 모드)                                              ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │ • LLM이 검색 결과를 기반으로 답변 생성                                   │  ║
║  │ • 자연스러운 문장으로 응답                                              │  ║
║  │ • Answer Instructions로 커스터마이징 가능                               │  ║
║  │ • 추가 토큰 비용 발생                                                   │  ║
║  │                                                                        │  ║
║  │ 적합한 경우:                                                            │  ║
║  │   - 챗봇/어시스턴트 응답                                                │  ║
║  │   - 요약이 필요한 경우                                                  │  ║
║  │   - 특정 형식의 답변 필요                                               │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

## 3. Output Mode 비교 실험

In [ ]:
def run_with_output_mode(
    query: str,
    output_mode: str,
    answer_instructions: str = None,
    top: int = 5
) -> dict:
    """
    지정된 Output Mode로 검색을 실행합니다.
    """
    # Output Mode 매핑
    mode_map = {
        "extractive": KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        "synthesis": KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS
    }
    
    # Answer Instructions 설정
    instructions = None
    if answer_instructions:
        instructions = KnowledgeRetrievalAnswerInstructions(
            instructions=answer_instructions
        )
    
    # 요청 생성
    request = KnowledgeBaseRetrievalRequest(
        query=query,
        top=top,
        retrieval_reasoning_effort=KnowledgeRetrievalMediumReasoningEffort(),
        output_mode=mode_map.get(output_mode, mode_map["extractive"]),
        answer_instructions=instructions
    )
    
    # 실행
    start_time = time.time()
    response = kb_client.retrieve(request)
    elapsed_time = time.time() - start_time
    
    return {
        "response": response,
        "elapsed_time": elapsed_time,
        "output_mode": output_mode
    }

print("✅ Output Mode 실행 함수 정의 완료")

In [ ]:
# 테스트 쿼리
TEST_QUERY = "겨울에 아기가 따뜻하게 입을 수 있는 옷을 추천해주세요"

print(f"📝 테스트 쿼리: {TEST_QUERY}")
print("\n" + "#" * 70)
print("🔬 Output Mode 비교 실험")
print("#" * 70)

In [ ]:
# Extractive Data 모드
result_extractive = run_with_output_mode(
    query=TEST_QUERY,
    output_mode="extractive",
    top=3
)

print("\n" + "=" * 60)
print("📊 EXTRACTIVE_DATA 모드 결과")
print("=" * 60)
print(f"⏱️  실행 시간: {result_extractive['elapsed_time']:.2f}초")

response = result_extractive['response']

# 답변 확인
if hasattr(response, 'answer') and response.answer:
    print(f"\n📝 답변: {response.answer}")
else:
    print("\n📝 답변: (생성되지 않음 - Extractive 모드)")

# 문서 확인
if hasattr(response, 'data') and response.data:
    print(f"\n📄 반환 문서 수: {len(response.data)}")
    for i, doc in enumerate(response.data[:3], 1):
        print(f"\n   [{i}] {doc.get('title', 'N/A')[:50]}")
        content = doc.get('content', '')[:200]
        print(f"       {content}...")

In [ ]:
# Answer Synthesis 모드
result_synthesis = run_with_output_mode(
    query=TEST_QUERY,
    output_mode="synthesis",
    top=3
)

print("\n" + "=" * 60)
print("📝 ANSWER_SYNTHESIS 모드 결과")
print("=" * 60)
print(f"⏱️  실행 시간: {result_synthesis['elapsed_time']:.2f}초")

response = result_synthesis['response']

# 답변 확인
if hasattr(response, 'answer') and response.answer:
    print(f"\n📝 생성된 답변:")
    print("-" * 50)
    print(response.answer)
    print("-" * 50)
else:
    print("\n⚠️ 답변이 생성되지 않았습니다.")

# 참조 문서 확인
if hasattr(response, 'data') and response.data:
    print(f"\n📚 참조 문서 수: {len(response.data)}")

In [ ]:
# 비교 요약
print("\n" + "=" * 70)
print("📊 Output Mode 비교 요약")
print("=" * 70)

print(f"\n{'항목':<25} {'Extractive':<20} {'Synthesis':<20}")
print("-" * 65)

# 실행 시간
print(f"{'실행 시간':<25} {result_extractive['elapsed_time']:.2f}초{'':<14} {result_synthesis['elapsed_time']:.2f}초")

# 답변 생성 여부
ext_has_answer = "✅" if (hasattr(result_extractive['response'], 'answer') and result_extractive['response'].answer) else "❌"
syn_has_answer = "✅" if (hasattr(result_synthesis['response'], 'answer') and result_synthesis['response'].answer) else "❌"
print(f"{'답변 생성':<25} {ext_has_answer:<20} {syn_has_answer:<20}")

# 토큰 사용량
ext_tokens = getattr(result_extractive['response'].usage, 'total_tokens', 'N/A') if hasattr(result_extractive['response'], 'usage') and result_extractive['response'].usage else 'N/A'
syn_tokens = getattr(result_synthesis['response'].usage, 'total_tokens', 'N/A') if hasattr(result_synthesis['response'], 'usage') and result_synthesis['response'].usage else 'N/A'
print(f"{'토큰 사용량':<25} {str(ext_tokens):<20} {str(syn_tokens):<20}")

## 4. Answer Instructions 커스터마이징

**Answer Instructions**를 통해 LLM의 답변 생성 방식을 세밀하게 제어할 수 있습니다.

In [ ]:
# 기본 Instructions 정의
INSTRUCTIONS = {
    "default": None,
    
    "concise": """
간결하게 핵심만 답변하세요.
- 3문장 이내로 요약
- 가격과 브랜드 정보 포함
- 불필요한 수식어 제외
""",
    
    "detailed": """
상세하고 친절하게 답변하세요.
- 각 제품의 특징을 자세히 설명
- 장단점 분석 포함
- 추천 이유 설명
- 관련 팁이나 조언 추가
""",
    
    "comparison": """
제품 비교표 형식으로 답변하세요.
- 표 형식으로 정리
- 가격, 브랜드, 특징 비교
- 최종 추천 제품 선정
""",
    
    "json_format": """
JSON 형식으로 답변하세요.
{
  "recommendations": [
    {
      "product_name": "제품명",
      "brand": "브랜드",
      "price": "가격",
      "key_features": ["특징1", "특징2"],
      "recommendation_reason": "추천 이유"
    }
  ],
  "summary": "전체 요약"
}
""",
    
    "bullet_points": """
불릿 포인트 형식으로 답변하세요.
• 각 제품을 항목별로 정리
• 중요도 순으로 나열
• 이모지 사용하여 가독성 향상
"""
}

print("✅ Answer Instructions 정의 완료")
print(f"\n📋 사용 가능한 Instructions: {list(INSTRUCTIONS.keys())}")

In [ ]:
# Instructions 비교 테스트
QUERY = "아기 겨울 옷 중에서 가성비 좋은 제품을 추천해주세요"

print(f"📝 테스트 쿼리: {QUERY}")
print("\n" + "#" * 70)
print("🎨 Answer Instructions 비교")
print("#" * 70)

In [ ]:
# 기본 (Instructions 없음)
result_default = run_with_output_mode(
    query=QUERY,
    output_mode="synthesis",
    answer_instructions=None,
    top=3
)

print("\n" + "=" * 60)
print("📋 기본 (Instructions 없음)")
print("=" * 60)
if hasattr(result_default['response'], 'answer') and result_default['response'].answer:
    print(result_default['response'].answer)
else:
    print("(답변 없음)")

In [ ]:
# 간결한 답변
result_concise = run_with_output_mode(
    query=QUERY,
    output_mode="synthesis",
    answer_instructions=INSTRUCTIONS["concise"],
    top=3
)

print("\n" + "=" * 60)
print("📋 간결한 답변 (Concise)")
print("=" * 60)
if hasattr(result_concise['response'], 'answer') and result_concise['response'].answer:
    print(result_concise['response'].answer)
else:
    print("(답변 없음)")

In [ ]:
# 상세한 답변
result_detailed = run_with_output_mode(
    query=QUERY,
    output_mode="synthesis",
    answer_instructions=INSTRUCTIONS["detailed"],
    top=3
)

print("\n" + "=" * 60)
print("📋 상세한 답변 (Detailed)")
print("=" * 60)
if hasattr(result_detailed['response'], 'answer') and result_detailed['response'].answer:
    print(result_detailed['response'].answer)
else:
    print("(답변 없음)")

In [ ]:
# JSON 형식 답변
result_json = run_with_output_mode(
    query=QUERY,
    output_mode="synthesis",
    answer_instructions=INSTRUCTIONS["json_format"],
    top=3
)

print("\n" + "=" * 60)
print("📋 JSON 형식 답변")
print("=" * 60)
if hasattr(result_json['response'], 'answer') and result_json['response'].answer:
    answer = result_json['response'].answer
    print(answer)
    
    # JSON 파싱 시도
    try:
        # JSON 블록 추출
        if "```json" in answer:
            json_str = answer.split("```json")[1].split("```")[0]
        elif "```" in answer:
            json_str = answer.split("```")[1].split("```")[0]
        else:
            json_str = answer
        
        parsed = json.loads(json_str.strip())
        print("\n✅ JSON 파싱 성공!")
    except:
        print("\n⚠️ JSON 파싱 실패 (형식 검토 필요)")
else:
    print("(답변 없음)")

In [ ]:
# 불릿 포인트 형식
result_bullets = run_with_output_mode(
    query=QUERY,
    output_mode="synthesis",
    answer_instructions=INSTRUCTIONS["bullet_points"],
    top=3
)

print("\n" + "=" * 60)
print("📋 불릿 포인트 형식")
print("=" * 60)
if hasattr(result_bullets['response'], 'answer') and result_bullets['response'].answer:
    print(result_bullets['response'].answer)
else:
    print("(답변 없음)")

## 5. 도메인별 커스터마이징

In [ ]:
# 도메인별 Answer Instructions
DOMAIN_INSTRUCTIONS = {
    "ecommerce_product": """
쇼핑몰 상품 추천 어시스턴트로서 답변하세요.

답변 형식:
🛍️ **추천 상품**
1. [상품명] - [가격]
   - 브랜드: [브랜드명]
   - 특징: [주요 특징 3개]
   - 추천 이유: [한 줄 설명]

💡 **구매 팁**
- 관련 구매 조언 제공

가격은 원화로 표시하고, 친근하고 전문적인 톤을 유지하세요.
""",
    
    "customer_service": """
고객 서비스 담당자로서 답변하세요.

지침:
- 공손하고 친절한 어조 사용
- "안녕하세요, 고객님"으로 시작
- 문제 해결 중심 답변
- 추가 도움이 필요하면 문의 안내
- "감사합니다"로 마무리
""",
    
    "technical_doc": """
기술 문서 작성자로서 답변하세요.

형식:
## 개요
[간략한 설명]

## 상세 내용
[기술적 세부사항]

## 참고 사항
[주의점 또는 추가 정보]

정확하고 객관적인 톤을 유지하세요.
""",
    
    "qa_bot": """
FAQ 챗봇으로서 답변하세요.

지침:
- 질문에 직접 답변 (예/아니오로 시작 가능)
- 간단명료하게 핵심만 전달
- 관련 추가 정보 1-2개 제공
- 200자 이내로 제한
"""
}

print("✅ 도메인별 Instructions 정의 완료")
print(f"\n📋 도메인 목록: {list(DOMAIN_INSTRUCTIONS.keys())}")

In [ ]:
# E-commerce 도메인 테스트
result_ecommerce = run_with_output_mode(
    query="겨울 아기 외출복 추천해주세요",
    output_mode="synthesis",
    answer_instructions=DOMAIN_INSTRUCTIONS["ecommerce_product"],
    top=3
)

print("\n" + "=" * 60)
print("🛍️ E-commerce 상품 추천 스타일")
print("=" * 60)
if hasattr(result_ecommerce['response'], 'answer') and result_ecommerce['response'].answer:
    print(result_ecommerce['response'].answer)

In [ ]:
# 고객 서비스 도메인 테스트
result_cs = run_with_output_mode(
    query="주문한 아기 옷 교환이 가능한가요?",
    output_mode="synthesis",
    answer_instructions=DOMAIN_INSTRUCTIONS["customer_service"],
    top=3
)

print("\n" + "=" * 60)
print("💬 고객 서비스 스타일")
print("=" * 60)
if hasattr(result_cs['response'], 'answer') and result_cs['response'].answer:
    print(result_cs['response'].answer)

In [ ]:
# QA Bot 도메인 테스트
result_qa = run_with_output_mode(
    query="블루독베이비 제품이 있나요?",
    output_mode="synthesis",
    answer_instructions=DOMAIN_INSTRUCTIONS["qa_bot"],
    top=3
)

print("\n" + "=" * 60)
print("🤖 QA Bot 스타일")
print("=" * 60)
if hasattr(result_qa['response'], 'answer') and result_qa['response'].answer:
    print(result_qa['response'].answer)

## 6. 고급 Answer Instructions 패턴

In [ ]:
# 고급 패턴: 조건부 응답
CONDITIONAL_INSTRUCTIONS = """
다음 조건에 따라 답변 형식을 선택하세요:

1. 검색 결과가 3개 이상인 경우:
   - 비교표 형식으로 정리
   - 가격 오름차순 정렬

2. 검색 결과가 1-2개인 경우:
   - 상세 설명 형식
   - 유사 제품 언급

3. 검색 결과가 없는 경우:
   - "죄송합니다, 해당 조건의 제품을 찾지 못했습니다."
   - 대안 제안

모든 경우에 한국어로 답변하세요.
"""

result_conditional = run_with_output_mode(
    query="3만원 이하 아기 겨울 코트",
    output_mode="synthesis",
    answer_instructions=CONDITIONAL_INSTRUCTIONS,
    top=5
)

print("\n" + "=" * 60)
print("🎯 조건부 응답 패턴")
print("=" * 60)
if hasattr(result_conditional['response'], 'answer') and result_conditional['response'].answer:
    print(result_conditional['response'].answer)

In [ ]:
# 고급 패턴: 페르소나 기반 응답
PERSONA_INSTRUCTIONS = """
당신은 "베이비 전문가 미미"입니다.

페르소나 특성:
- 10년 경력의 육아 전문가
- 친근하고 따뜻한 말투 ("~해요", "~예요" 사용)
- 이모지 적극 활용 👶🍼💕
- 실용적인 조언 제공
- "미미의 팁!" 코너 포함

답변 구조:
1. 인사 ("안녕하세요, 미미예요!")
2. 공감 표현
3. 제품 추천
4. 미미의 팁
5. 마무리 인사
"""

result_persona = run_with_output_mode(
    query="신생아 외출할 때 뭘 입혀야 할지 모르겠어요",
    output_mode="synthesis",
    answer_instructions=PERSONA_INSTRUCTIONS,
    top=3
)

print("\n" + "=" * 60)
print("🎭 페르소나 기반 응답 (베이비 전문가 미미)")
print("=" * 60)
if hasattr(result_persona['response'], 'answer') and result_persona['response'].answer:
    print(result_persona['response'].answer)

In [ ]:
# 고급 패턴: 다국어 지원
MULTILINGUAL_INSTRUCTIONS = """
사용자의 질문 언어에 맞춰 답변하세요.

한국어 질문 → 한국어 답변
English question → English answer
日本語の質問 → 日本語で回答

답변 형식:
- 간결하게 3-5문장
- 제품명은 원문 유지
- 가격은 현지 통화로 표시
"""

print("\n" + "=" * 60)
print("🌏 다국어 지원 패턴")
print("=" * 60)
print("(다국어 Answer Instructions 정의 완료)")

## 7. 토큰 효율성 비교

In [ ]:
# 토큰 사용량 비교
print("\n" + "=" * 70)
print("💰 Answer Instructions에 따른 토큰 사용량 비교")
print("=" * 70)

results = [
    ("기본 (No Instructions)", result_default),
    ("간결한 답변", result_concise),
    ("상세한 답변", result_detailed),
    ("JSON 형식", result_json),
    ("불릿 포인트", result_bullets)
]

print(f"\n{'스타일':<25} {'실행시간':<12} {'토큰':<12} {'답변 길이':<12}")
print("-" * 65)

for name, result in results:
    elapsed = f"{result['elapsed_time']:.2f}초"
    tokens = getattr(result['response'].usage, 'total_tokens', 'N/A') if hasattr(result['response'], 'usage') and result['response'].usage else 'N/A'
    answer_len = len(result['response'].answer) if hasattr(result['response'], 'answer') and result['response'].answer else 0
    
    print(f"{name:<25} {elapsed:<12} {str(tokens):<12} {answer_len}자")

## 8. Best Practices

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                   📋 Answer Instructions Best Practices                       ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  ✅ 권장 사항                                                                  ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │ 1. 명확한 형식 지정                                                     │  ║
║  │    - 구체적인 출력 형식 예시 제공                                        │  ║
║  │    - 항목 수, 길이 제한 명시                                            │  ║
║  │                                                                        │  ║
║  │ 2. 톤/어조 정의                                                         │  ║
║  │    - 타겟 사용자에 맞는 어조 설정                                        │  ║
║  │    - 전문적/친근한/공손한 등 명시                                        │  ║
║  │                                                                        │  ║
║  │ 3. 조건부 로직 포함                                                     │  ║
║  │    - 검색 결과에 따른 분기 처리                                          │  ║
║  │    - 예외 상황 대응 지침                                                │  ║
║  │                                                                        │  ║
║  │ 4. 일관성 유지                                                          │  ║
║  │    - 모든 응답에서 동일한 구조 유지                                      │  ║
║  │    - 용어 통일                                                          │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
║  ❌ 피해야 할 것                                                               ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │ 1. 너무 긴 Instructions (토큰 낭비)                                     │  ║
║  │ 2. 모호한 지시사항                                                      │  ║
║  │ 3. 검색 결과와 무관한 정보 요청                                          │  ║
║  │ 4. 복잡한 계산이나 로직 요구                                             │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
║  🎯 사용 사례별 권장 Output Mode                                               ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │ RAG 파이프라인 백엔드      → EXTRACTIVE_DATA                           │  ║
║  │ 챗봇/어시스턴트 응답       → ANSWER_SYNTHESIS + Instructions           │  ║
║  │ 데이터 분석/정리           → ANSWER_SYNTHESIS + JSON Instructions      │  ║
║  │ 빠른 검색 결과 반환        → EXTRACTIVE_DATA                           │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

## 9. 정리

이 노트북에서 학습한 내용:

1. **Output Mode**: Extractive Data vs Answer Synthesis 비교
2. **Answer Instructions**: 답변 형식 및 스타일 커스터마이징
3. **도메인별 설정**: E-commerce, 고객서비스, QA Bot 등
4. **고급 패턴**: 조건부 응답, 페르소나, 다국어 지원
5. **토큰 효율성**: Instructions에 따른 비용 최적화

In [ ]:
print("\n" + "=" * 70)
print("🎉 모든 Advanced FoundryIQ 핸즈온 노트북 완료!")
print("=" * 70)
print("""
📚 완료한 Advanced 노트북 목록:

  04-multi_source_kb.ipynb      ✅ Multi-Source Knowledge Base
  05-runtime_parameters.ipynb   ✅ Runtime Parameters 튜닝
  06-mcp_integration.ipynb      ✅ MCP Integration
  07-activity_tracing.ipynb     ✅ Activity Tracing 및 디버깅
  08-answer_synthesis.ipynb     ✅ Answer Synthesis 및 Output Mode

🔗 참고 자료:
  - Azure AI Search Knowledge Retrieval Demo:
    https://github.com/Azure-Samples/azure-ai-search-knowledge-retrieval-demo
  - Azure Search OpenAI Demo:
    https://github.com/Azure-Samples/azure-search-openai-demo

💡 다음 단계:
  - 프로덕션 배포 준비 (10-deploy_demo/)
  - 실제 데이터로 테스트
  - 성능 모니터링 설정
""")